# TUIA NLP 2026 — Trabajo Práctico Final
## Asistente Virtual de Electrodomésticos — ElectroHogar

**Alumno:** Maximiliano Francisco Chiappe  
**Repositorio:** https://github.com/MaxiChiappe/TP_NLP_CHIAPPE_Maximiliano.git

---

### Estructura del notebook

**Celdas 1-3:** Instalación, imports y carga de datos

**Ejercicio 1 — RAG:**
- Item 1.8: Modelo de lenguaje local (Celdas 4-5)
- Item 1.1: Diseño de fuentes de datos (Markdown)
- Item 1.2: Base de datos vectorial (Celdas 6-7)
- Item 1.3: Base de datos tabular (Celdas 8-9)
- Item 1.4: Base de datos de grafos (Celdas 10-14)
- Item 1.5: Clasificadores de intención (Celdas 15-17)
- Item 1.6: Búsqueda híbrida + ReRank (Celda 18)
- Item 1.7: Checkpoint obligatorio (Celda 19)
- Item 1.9: Chatbot con memoria (Celda 20)

**Ejercicio 2 — Agente ReAct:** Celdas 21-24

##Ejercicio 1 - RAG

In [1]:
# CELDA 1: Instalación de dependencias base.
#Nota: transformers y neo4j se instalan más adelante cuando los necesitemos para no hacer la celda inicial muy pesada
!pip install -q chromadb 2>/dev/null
!pip install -q sentence-transformers 2>/dev/null
!pip install -q rank-bm25 2>/dev/null
!pip install -q langchain langchain-community langchain-core 2>/dev/null
!pip install -q scikit-learn 2>/dev/null

print("Dependencias base instaladas")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 83.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 127.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 87.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 

In [2]:
# CELDA 2: Librerías necesarias para el proyecto

import os
import json
import zipfile
import pandas as pd
import numpy as np
from pathlib import Path

# LangChain
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings y vectorstore
import chromadb
from sentence_transformers import SentenceTransformer

# BM25
from rank_bm25 import BM25Okapi

# Clasificador de intención
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report

print("Imports OK")

Imports OK


In [3]:
# CELDA 3: Descarga del dataset

# Descargo el dataset desde el repositorio de la cátedra
# Uso wget con el link de GitHub porque no requiere autenticación
!wget -q https://github.com/jpmanson/tuia-unr/raw/refs/heads/main/nlp/fuentes_de_informacion-tp-nlp-2026.zip -O /content/fuentes.zip
!unzip -q -o /content/fuentes.zip -d /content/

BASE = "/content/fuentes_de_informacion/"

# Cargo los CSVs como DataFrames de pandas
df_productos    = pd.read_csv(BASE + "productos.csv")
df_vendedores   = pd.read_csv(BASE + "vendedores.csv")
df_ventas       = pd.read_csv(BASE + "ventas_historicas.csv")
df_tickets      = pd.read_csv(BASE + "tickets_soporte.csv")
df_inventario   = pd.read_csv(BASE + "inventario_sucursales.csv")
df_devoluciones = pd.read_csv(BASE + "devoluciones.csv")

# El JSON de FAQs lo convierto directamente a DataFrame
with open(BASE + "faqs.json", "r", encoding="utf-8") as f:
    faqs = json.load(f)
df_faqs = pd.DataFrame(faqs)

# Para manuales y reseñas solo guardo las rutas
# Los leo archivo por archivo cuando indexe en ChromaDB
MANUALES_DIR = Path(BASE + "manuales_productos")
RESENAS_DIR  = Path(BASE + "resenas_usuarios")

# Verifico que todo cargó bien
print(f"Productos:    {len(df_productos)} filas")
print(f"Ventas:       {len(df_ventas)} filas")
print(f"Inventario:   {len(df_inventario)} filas")
print(f"Tickets:      {len(df_tickets)} filas")
print(f"Devoluciones: {len(df_devoluciones)} filas")
print(f"FAQs:         {len(df_faqs)} filas")
print(f"Manuales:     {len(list(MANUALES_DIR.glob('*.md')))} archivos")
print(f"Reseñas:      {len(list(RESENAS_DIR.glob('*.txt')))} archivos")

Productos:    300 filas
Ventas:       10000 filas
Inventario:   4100 filas
Tickets:      2000 filas
Devoluciones: 800 filas
FAQs:         3000 filas
Manuales:     50 archivos
Reseñas:      5015 archivos


### 1.8 Modelo de Lenguaje Local

Modelo utilizado: **Qwen 2.5 3B Instruct** (Hugging Face, ejecución local con `transformers`).

Se carga antes de las bases de datos porque se usa en múltiples componentes: generación de SQL, Cypher y clasificador Few-Shot.

In [4]:
# CELDA 4: Instalación de las librerías para el LLM local

!pip install -q transformers accelerate bitsandbytes

print("Librerías instaladas")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.7 MB/s eta 0:00:00
Librerías instaladas


In [5]:
# CELDA 5: Cargar modelo Qwen 2.5 3B

import warnings
warnings.filterwarnings("ignore")

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODELO = "Qwen/Qwen2.5-3B-Instruct"
MODELO_DIR = "/content/modelo_llm"

print("Cargando modelo...")

# Descarga el modelo de Hugging Face y lo cachea en MODELO_DIR
tokenizer = AutoTokenizer.from_pretrained(
    MODELO,
    cache_dir=MODELO_DIR
)

modelo_llm = AutoModelForCausalLM.from_pretrained(
    MODELO,
    cache_dir=MODELO_DIR,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Modelo cargado!")
print(f"Dispositivo: {next(modelo_llm.parameters()).device}")

def generar(prompt_sistema, prompt_usuario, max_tokens=256):
    """
    Genera una respuesta usando el formato de chat de Qwen 2.5.

    prompt_sistema : instrucciones para el modelo (rol, restricciones)
    prompt_usuario : la pregunta o tarea concreta
    max_tokens     : máxima cantidad de tokens a generar
    """
    messages = [
        {"role": "system", "content": prompt_sistema},
        {"role": "user",   "content": prompt_usuario}
    ]

    texto = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(texto, return_tensors="pt").to(modelo_llm.device)

    with torch.no_grad():
        outputs = modelo_llm.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.3
        )

    respuesta = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return respuesta.strip()

# Prueba
respuesta = generar(
    prompt_sistema="Sos un asistente conciso. Respondé en una sola oración corta.",
    prompt_usuario="¿Cuál es la capital de Argentina?"
)
print(f"Respuesta: {respuesta}")

Cargando modelo...


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Modelo cargado!
Dispositivo: cuda:0
Respuesta: Buenos Aires


### 1.1 Diseño de las Fuentes de Datos

---

### Base de Datos Vectorial (ChromaDB)
**Información almacenada:** descripciones de productos, FAQs, manuales técnicos y reseñas de usuarios.

**Justificación:** estas fuentes contienen texto no estructurado donde el significado semántico importa más que la coincidencia exacta de palabras. La búsqueda vectorial por similitud coseno permite encontrar documentos relevantes aunque el usuario use palabras distintas a las del documento.

**Estructura:** 9.456 chunks de 500 caracteres con 50 de superposición, con metadata de fuente (faq, manual, resena, prod) para filtrado posterior.

---

### Base de Datos Tabular (SQLite)
**Información almacenada:** productos, precios, stock, ventas históricas, devoluciones y vendedores.

**Justificación:** información estructurada y numérica que requiere filtros exactos por precio, stock y fechas. SQL maneja estas consultas con condiciones WHERE, rangos y agregaciones de forma eficiente.

**Estructura:** 5 tablas (productos, inventario, ventas, devoluciones, vendedores) con el esquema original de los CSVs.

---

### Base de Datos de Grafos (Neo4j)
**Información almacenada:** relaciones entre productos, categorías, subcategorías, marcas, sucursales y vendedores.

**Justificación:** las consultas sobre relaciones entre entidades se modelan naturalmente en un grafo, permitiendo navegarlas sin JOINs complejos.

**Estructura:** nodos Producto, Categoria, Subcategoria, Marca, Sucursal y Vendedor conectados por relaciones PERTENECE_A, ES_DE_MARCA, DISPONIBLE_EN y TRABAJA_EN.

### 1.2 Base de Datos Vectorial

**Modelo de embedding:** `paraphrase-multilingual-MiniLM-L12-v2`
- Soporte nativo para español
- Liviano (12 capas), corre eficientemente en la GPU T4 de Colab
- Vectores de 384 dimensiones

**Text Splitter:** `RecursiveCharacterTextSplitter`
- `chunk_size=500`: fragmentos de hasta 500 caracteres
- `chunk_overlap=50`: superposición de 50 caracteres para no perder contexto en los bordes

**Fuentes indexadas:** descripciones de productos, FAQs, manuales y reseñas — 9.456 chunks en total

In [6]:
# CELDA 6: Base de datos vectorial (ChromaDB)

import shutil

# ChromaDB guarda los datos localmente en Colab
CHROMA_LOCAL = "/content/chroma_db"

# Cargo el modelo de embeddings multilingüe
print("Cargando modelo de embeddings...")
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
print("Modelo cargado")


chroma_client = chromadb.PersistentClient(path=CHROMA_LOCAL)

# Borro la colección anterior para evitar datos viejos en caché
try:
    chroma_client.delete_collection("electrohogar")
    print("Colección anterior borrada")
except:
    pass

# Inicializo ChromaDB con similitud coseno (más apropiada para embeddings de texto)
collection = chroma_client.get_or_create_collection(
    name="electrohogar",
    metadata={"hnsw:space": "cosine"}
)

if collection.count() > 0:
    print(f"ChromaDB ya indexado ({collection.count()} chunks). Saltando indexación.")
else:
    print("Primera vez — indexando todo...")


    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

    def agregar_documentos(textos, metadatas, prefijo):
        chunks_totales = []
        metas_totales  = []
        ids_totales    = []

        for i, (texto, meta) in enumerate(zip(textos, metadatas)):
            chunks = splitter.split_text(texto)
            for j, chunk in enumerate(chunks):
                chunks_totales.append(chunk)
                metas_totales.append(meta)
                ids_totales.append(f"{prefijo}_{i}_{j}")

        # batch_size=64 para aprovechar la GPU eficientemente
        embeddings = embedding_model.encode(chunks_totales, batch_size=64, show_progress_bar=True)

        collection.add(
            documents=chunks_totales,
            embeddings=embeddings.tolist(),
            metadatas=metas_totales,
            ids=ids_totales
        )
        print(f"{len(chunks_totales)} chunks agregados con prefijo '{prefijo}'")

    # Indexo cada fuente por separado con su metadata de "fuente" para poder filtrar por tipo de documento después
    print("\n Indexando productos...")
    textos_productos = df_productos["descripcion"].fillna("").tolist()
    metas_productos  = df_productos[["id_producto", "nombre", "categoria", "precio_usd"]].to_dict("records")
    agregar_documentos(textos_productos, metas_productos, "prod")

    print("\n Indexando FAQs...")
    textos_faqs = (df_faqs["pregunta"] + " " + df_faqs["respuesta"]).tolist()
    df_faqs_meta = df_faqs[["id_faq", "id_producto", "nombre_producto", "categoria"]].copy()
    df_faqs_meta["fuente"] = "faq"
    metas_faqs = df_faqs_meta.to_dict("records")
    agregar_documentos(textos_faqs, metas_faqs, "faq")

    print("\n Indexando manuales...")
    textos_manuales = []
    metas_manuales  = []
    for archivo in MANUALES_DIR.glob("*.md"):
        texto = archivo.read_text(encoding="utf-8")
        id_producto = archivo.stem.split("_")[1]
        textos_manuales.append(texto)
        metas_manuales.append({"fuente": "manual", "id_producto": id_producto, "archivo": archivo.name})
    agregar_documentos(textos_manuales, metas_manuales, "man")

    print("\n Indexando reseñas...")
    textos_resenas = []
    metas_resenas  = []
    for archivo in RESENAS_DIR.glob("*.txt"):
        texto = archivo.read_text(encoding="utf-8")
        textos_resenas.append(texto)
        metas_resenas.append({"fuente": "resena", "archivo": archivo.name})
    agregar_documentos(textos_resenas, metas_resenas, "res")

    print("\n Base vectorial completa")

print(f"\n Total chunks en ChromaDB: {collection.count()}")

Cargando modelo de embeddings...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

3000 chunks agregados con prefijo 'faq'

 Indexando manuales...


Batches:   0%|          | 0/13 [00:00<?, ?it/s]

828 chunks agregados con prefijo 'man'

 Indexando reseñas...


Batches:   0%|          | 0/79 [00:00<?, ?it/s]

5015 chunks agregados con prefijo 'res'

 Base vectorial completa

 Total chunks en ChromaDB: 9456


In [7]:
# CELDA 7: Función de búsqueda vectorial

def buscar_vectorial(consulta, k=5, filtro_fuente=None):
    """
    Busca los K chunks más similares a la consulta en ChromaDB.
    consulta: texto de la pregunta del usuario
    k: cantidad de resultados a devolver
    filtro_fuente : "manual", "resena", "faq" o None para buscar en todo

    """
    # Convierto la consulta en embedding
    embedding_consulta = embedding_model.encode(consulta).tolist()

    # Filtro opcional por fuente de datos
    where = {"fuente": filtro_fuente} if filtro_fuente else None

    resultados = collection.query(
        query_embeddings=[embedding_consulta],
        n_results=k,
        where=where
    )

    documentos = resultados["documents"][0]
    metadatas  = resultados["metadatas"][0]
    distancias = resultados["distances"][0]

    salida = []
    for doc, meta, dist in zip(documentos, metadatas, distancias):
        # ChromaDB devuelve distancia coseno (0 a 2)
        # La convierto a score de similitud (0 a 1) donde 1 = idéntico
        score = round(1 - dist, 4)
        salida.append({
            "contenido" : doc,
            "metadata"  : meta,
            "score"     : score
        })

    return salida

# Prueba con uno de los prompts obligatorios del checkpoint
resultados = buscar_vectorial("¿Cómo uso mi licuadora para hacer smoothies?", k=3)
for r in resultados:
    print(f"Score: {r['score']} | Fuente: {r['metadata'].get('fuente')} | Archivo: {r['metadata'].get('archivo', r['metadata'].get('nombre'))}")
    print(f"Contenido: {r['contenido'][:150]}...")
    print()

Score: 0.5424 | Fuente: faq | Archivo: None
Contenido: ¿Cuál es la forma correcta de encenderlo? El Mixer de KitchenPro está diseñado para uso doméstico. Revise el manual del producto (código P0025) para m...

Score: 0.5424 | Fuente: faq | Archivo: None
Contenido: ¿Cuál es la forma correcta de encenderlo? El Mixer de KitchenPro está diseñado para uso doméstico. Revise el manual del producto (código P0025) para m...

Score: 0.5178 | Fuente: faq | Archivo: None
Contenido: ¿Necesita algún tipo de lubricación? El Mixer Pro de ChefMaster está diseñado para uso doméstico. Revise el manual del producto (código P0027) para má...



### 1.3 Base de Datos Tabular

5 tablas en SQLite en memoria: productos, inventario, ventas, devoluciones y vendedores.

El LLM genera las consultas SQL dinámicamente usando como referencia el esquema de las tablas (columnas, valores únicos y rangos numéricos) sin necesidad de ver todos los datos.

In [8]:
# CELDA 8: Base de datos tabular (SQLite)

import sqlite3

# Creo la base de datos en memoria RAM — más rápida que en disco
# Se pierden los datos al reiniciar pero como los cargamos desde CSV no es problema
conn = sqlite3.connect(":memory:")

# Convierto cada DataFrame en una tabla SQL
df_productos.to_sql("productos", conn, if_exists="replace", index=False)
df_inventario.to_sql("inventario", conn, if_exists="replace", index=False)
df_ventas.to_sql("ventas", conn, if_exists="replace", index=False)
df_devoluciones.to_sql("devoluciones", conn, if_exists="replace", index=False)
df_vendedores.to_sql("vendedores", conn, if_exists="replace", index=False)

print("Tablas cargadas en SQLite")

def get_schema_info():
    """
    Genera un resumen del esquema de las tablas para pasarle al LLM.

    """
    info = []
    tablas = {
        "productos"   : df_productos,
        "inventario"  : df_inventario,
        "ventas"      : df_ventas,
        "devoluciones": df_devoluciones,
        "vendedores"  : df_vendedores
    }

    for nombre, df in tablas.items():
        info.append(f"\nTabla: {nombre}")
        info.append(f"Columnas: {', '.join(df.columns.tolist())}")

        # Muestro algunos valores únicos de columnas de texto para orientar al LLM
        for col in df.select_dtypes(include="object").columns[:4]:
            unicos = df[col].dropna().unique()[:8]
            info.append(f"  {col}: {list(unicos)}")

        # Muestro el rango de columnas numéricas
        for col in df.select_dtypes(include="number").columns[:4]:
            info.append(f"  {col}: min={df[col].min():.2f}, max={df[col].max():.2f}")

    return "\n".join(info)

schema_info = get_schema_info()
print("\n=== SCHEMA INFO (para el LLM) ===")
print(schema_info[:1000], "...")

def buscar_tabular(query_sql):
    """Ejecuta el SQL generado por el LLM y devuelve los resultados como DataFrame."""
    try:
        resultado = pd.read_sql_query(query_sql, conn)
        return resultado
    except Exception as e:
        return f"Error en consulta SQL: {e}"

# Prueba manual para verificar que las consultas funcionan
print("\n=== PRUEBA: licuadoras de menos de $500 ===")
sql = "SELECT nombre, precio_usd, stock FROM productos WHERE LOWER(nombre) LIKE '%licuadora%' AND precio_usd < 500 LIMIT 5"
print(buscar_tabular(sql))

Tablas cargadas en SQLite

=== SCHEMA INFO (para el LLM) ===

Tabla: productos
Columnas: id_producto, nombre, categoria, subcategoria, marca, precio_usd, stock, color, potencia_w, capacidad, voltaje, peso_kg, garantia_meses, descripcion
  id_producto: ['P0001', 'P0002', 'P0003', 'P0004', 'P0005', 'P0006', 'P0007', 'P0008']
  nombre: ['Licuadora', 'Plus Licuadora Pro', 'Compacto Licuadora', 'Ultra Licuadora', 'Procesadora', 'Deluxe Procesadora', 'Profesional Procesadora 2024', 'Advanced Procesadora II']
  categoria: ['Cocina', 'Climatización', 'Lavado', 'Audio y Video']
  subcategoria: ['Preparación', 'Cocción', 'Refrigeración', 'Pequeños Electrodomésticos', 'Aires Acondicionados', 'Calefacción', 'Ventilación', 'Purificación']
  precio_usd: min=28.22, max=2992.33
  stock: min=1.00, max=200.00
  potencia_w: min=40.00, max=1950.00
  peso_kg: min=0.60, max=49.40

Tabla: inventario
Columnas: id_inventario, sucursal, id_producto, nombre_producto, categoria, marca, stock_sucursal, stock_minim

In [9]:
# CELDA 9: Generación dinámica de SQL con LLM

import re

def limpiar_sql(texto):
    """Limpia la salida del LLM — a veces agrega markdown o backticks innecesarios."""
    texto = re.sub(r"```sql|```", "", texto).strip()
    texto = texto.replace("`", "")
    texto = texto.strip().split(";")[0] + ";"
    return texto

def generar_sql(consulta_usuario):
    """
    Convierte lenguaje natural a SQL usando el LLM con few-shot prompting.
    Si el SQL generado falla, usa un sistema de fallback con palabras clave.

    """
    prompt_sistema = """Sos un experto en SQL para SQLite. Generá SOLO la consulta SQL, sin explicaciones ni markdown.

Tablas disponibles:
- productos: id_producto, nombre, categoria, subcategoria, marca, precio_usd, stock, potencia_w, capacidad, voltaje, peso_kg, garantia_meses
- ventas: id_venta, fecha, id_producto, nombre_producto, id_vendedor, sucursal, cantidad, precio_unitario, descuento_pct, total, metodo_pago
- devoluciones: id_devolucion, id_venta, fecha_devolucion, id_producto, nombre_producto, motivo, estado, monto_reembolso
- vendedores: id_vendedor, nombre, apellido, sucursal, nivel, activo

EJEMPLOS:
Pregunta: ¿Qué licuadoras tienen disponibles?
SQL: SELECT nombre, precio_usd, stock FROM productos WHERE LOWER(nombre) LIKE '%licuadora%' AND stock > 0 LIMIT 10;

Pregunta: ¿Cuál es la licuadora más barata?
SQL: SELECT nombre, precio_usd, stock FROM productos WHERE LOWER(nombre) LIKE '%licuadora%' ORDER BY precio_usd ASC LIMIT 1;

Pregunta: ¿Cuáles son las licuadoras de menos de $500?
SQL: SELECT nombre, precio_usd, stock FROM productos WHERE LOWER(nombre) LIKE '%licuadora%' AND precio_usd < 500 LIMIT 10;

Pregunta: ¿Qué cafeteras hay disponibles?
SQL: SELECT nombre, precio_usd, stock FROM productos WHERE LOWER(nombre) LIKE '%cafetera%' AND stock > 0 LIMIT 10;

Pregunta: ¿Cuántas ventas se hicieron en total?
SQL: SELECT COUNT(*) as total_ventas FROM ventas;

Pregunta: ¿Qué productos tienen más de 100 unidades en stock?
SQL: SELECT nombre, stock FROM productos WHERE stock > 100 ORDER BY stock DESC LIMIT 10;

Pregunta: ¿Cuál es el producto más barato?
SQL: SELECT nombre, precio_usd FROM productos ORDER BY precio_usd ASC LIMIT 1;"""

    # Le paso ejemplos concretos (few-shot) para que el modelo aprenda el estilo de consultas que necesito sin inventar tablas o columnas
    prompt_usuario = f"Pregunta: {consulta_usuario}\nSQL:"

    sql_raw = generar(prompt_sistema, prompt_usuario, max_tokens=80)
    sql = limpiar_sql(sql_raw)

    # Valido el SQL ejecutándolo y si falla activo el fallback
    resultado = buscar_tabular(sql)
    if isinstance(resultado, str) and "Error" in resultado:
        # Fallback: construyo el SQL manualmente según palabras clave
        consulta_lower = consulta_usuario.lower()
        productos_keywords = {
            "licuadora": "licuadora", "cafetera": "cafetera", "heladera": "heladera",
            "microondas": "microondas", "freidora": "freidora", "plancha": "plancha",
            "lavarropas": "lavarropas", "televisor": "televisor", "freezer": "freezer"
        }

        for palabra, like in productos_keywords.items():
            if palabra in consulta_lower:
                if "barata" in consulta_lower or "económica" in consulta_lower or "menos" in consulta_lower:
                    sql = f"SELECT nombre, precio_usd, stock FROM productos WHERE LOWER(nombre) LIKE '%{like}%' ORDER BY precio_usd ASC LIMIT 5;"
                elif "cara" in consulta_lower or "más costosa" in consulta_lower:
                    sql = f"SELECT nombre, precio_usd, stock FROM productos WHERE LOWER(nombre) LIKE '%{like}%' ORDER BY precio_usd DESC LIMIT 5;"
                else:
                    sql = f"SELECT nombre, precio_usd, stock FROM productos WHERE LOWER(nombre) LIKE '%{like}%' AND stock > 0 LIMIT 10;"
                break

        if "cuántas ventas" in consulta_lower or "total de ventas" in consulta_lower:
            sql = "SELECT COUNT(*) as total_ventas FROM ventas;"
        elif "ventas" in consulta_lower:
            sql = "SELECT COUNT(*) as total_ventas FROM ventas;"
        elif "stock" in consulta_lower and "más" in consulta_lower:
            sql = "SELECT nombre, stock FROM productos WHERE stock > 100 ORDER BY stock DESC LIMIT 10;"

    return sql

# Pruebo con consultas representativas
consultas = [
    "¿Qué licuadoras tienen disponibles?",
    "¿Cuál es la licuadora más barata?",
    "¿Cuántas ventas se hicieron en total?",
]

for c in consultas:
    print(f"\nConsulta: {c}")
    sql = generar_sql(c)
    print(f"SQL: {sql}")
    resultado = buscar_tabular(sql)
    if isinstance(resultado, pd.DataFrame):
        print(resultado.head(3).to_string(index=False))
    else:
        print(f"Error: {resultado}")


Consulta: ¿Qué licuadoras tienen disponibles?
SQL: SELECT nombre, precio_usd, stock FROM productos WHERE LOWER(nombre) LIKE '%licuadora%' AND stock > 0 LIMIT 10;
            nombre  precio_usd  stock
         Licuadora      283.63    108
         Licuadora     1273.06    114
Plus Licuadora Pro      329.07     97

Consulta: ¿Cuál es la licuadora más barata?
SQL: SELECT nombre, precio_usd, stock FROM productos WHERE LOWER(nombre) LIKE '%licuadora%' ORDER BY precio_usd ASC LIMIT 1;
            nombre  precio_usd  stock
Compacto Licuadora      259.42     75

Consulta: ¿Cuántas ventas se hicieron en total?
SQL: SELECT COUNT(id_venta) AS total_ve FROM ventas;
 total_ve
    10000


### 1.4 Base de Datos de Grafos (Neo4j)

Relaciones modeladas en el grafo:
- `(:Producto)-[:PERTENECE_A]->(:Categoria)`
- `(:Producto)-[:PERTENECE_A]->(:Subcategoria)`
- `(:Producto)-[:ES_DE_MARCA]->(:Marca)`
- `(:Vendedor)-[:TRABAJA_EN]->(:Sucursal)`
- `(:Producto)-[:DISPONIBLE_EN]->(:Sucursal)`

Las consultas se generan dinámicamente en **Cypher** usando el LLM con few-shot prompting.

In [10]:
# CELDA 10: Instalación Neo4j local en Colab

# Instalo el driver Python PRIMERO antes de descargar el servidor
!pip install -q neo4j

# Instalo Java
!apt-get install -y openjdk-11-jdk -q

# Borro carpeta neo4j anterior si existe para evitar conflictos
import shutil, os
if os.path.exists("/content/neo4j"):
    shutil.rmtree("/content/neo4j")

# Descargo y descomprimo el servidor
!wget -q https://dist.neo4j.org/neo4j-community-5.13.0-unix.tar.gz
!tar -xf neo4j-community-5.13.0-unix.tar.gz
!mv neo4j-community-5.13.0 neo4j

# Desactivo autenticación
config = "dbms.security.auth_enabled=false\n"
with open("neo4j/conf/neo4j.conf", "a") as f:
    f.write(config)

print("Neo4j instalado")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 7.6 MB/s eta 0:00:00
Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  at-spi2-core fonts-dejavu-core fonts-dejavu-extra gsettings-desktop-schemas
  libatk-bridge2.0-0 libatk-wrapper-java libatk-wrapper-java-jni libatk1.0-0
  libatk1.0-data libatspi2.0-0 libxcomposite1 libxt-dev libxtst6 libxxf86dga1
  openjdk-11-jdk-headless openjdk-11-jre openjdk-11-jre-headless
  session-migration x11-utils
Suggested packages:
  libxt-doc openjdk-11-demo openjdk-11-source visualvm libnss-mdns
  fonts-ipafont-gothic fonts-ipafont-mincho fonts-wqy-microhei
  | fonts-wqy-zenhei fonts-indic mesa-utils
The following NEW packages will be installed:
  at-spi2-core fonts-dejavu-core fonts-dejavu-extra gsettings-desktop-schemas
  libatk-bridge2.0-0 libatk-wrapper-java libatk-wrapper-java-jni libatk1.0-0
  libatk1.0-data libatspi2.0-0 libxcomposite1 libxt-dev libxts

In [11]:
# CELDA 11: Iniciar Neo4j

import subprocess
import time

# Inicio Neo4j en segundo plano
!neo4j/bin/neo4j start

# Espero 15 segundos para que termine de arrancar antes de conectarme
print("Esperando que Neo4j arranque...")
time.sleep(15)

# Verifico que está corriendo
result = !neo4j/bin/neo4j status
print(result)

# Me conecto usando el protocolo Bolt en el puerto 7687 (puerto por defecto de Neo4j)
# auth=None porque desactivé la autenticación en la celda anterior
from neo4j import GraphDatabase

driver = GraphDatabase.driver("bolt://localhost:7687", auth=None)

# Prueba mínima para verificar que la conexión funciona
with driver.session() as session:
    result = session.run("RETURN 1 AS test")
    print("Neo4j conectado:", result.single()["test"])

Directories in use:
home:         /content/neo4j
config:       /content/neo4j/conf
logs:         /content/neo4j/logs
plugins:      /content/neo4j/plugins
import:       /content/neo4j/import
data:         /content/neo4j/data
certificates: /content/neo4j/certificates
licenses:     /content/neo4j/licenses
run:          /content/neo4j/run
Starting Neo4j.
Started neo4j (pid:8790). It is available at http://localhost:7474
There may be a short delay until the server is ready.
Esperando que Neo4j arranque...
['Neo4j is running at pid 8790']
Neo4j conectado: 1


In [12]:
# CELDA 12: Cargar relaciones en Neo4j

def cargar_grafo():
    with driver.session() as session:

        # Borro el grafo anterior para evitar datos duplicados
        session.run("MATCH (n) DETACH DELETE n")
        print("Grafo limpio")

        # Creo nodos y relaciones para productos, categorías, subcategorías y marcas
        # MERGE crea el nodo solo si no existe (evita duplicados)
        for _, row in df_productos.iterrows():
            session.run("""
                MERGE (p:Producto {id: $id, nombre: $nombre, precio: $precio})
                MERGE (c:Categoria {nombre: $categoria})
                MERGE (s:Subcategoria {nombre: $subcategoria})
                MERGE (m:Marca {nombre: $marca})
                MERGE (p)-[:PERTENECE_A]->(c)
                MERGE (p)-[:PERTENECE_A]->(s)
                MERGE (p)-[:ES_DE_MARCA]->(m)
            """, {
                "id"          : row["id_producto"],
                "nombre"      : row["nombre"],
                "precio"      : row["precio_usd"],
                "categoria"   : row["categoria"],
                "subcategoria": row["subcategoria"],
                "marca"       : row["marca"]
            })
        print("Productos, categorías y marcas cargados")

        # Creo nodos de vendedores y sucursales con su relación
        for _, row in df_vendedores.iterrows():
            session.run("""
                MERGE (v:Vendedor {id: $id, nombre: $nombre})
                MERGE (s:Sucursal {nombre: $sucursal})
                MERGE (v)-[:TRABAJA_EN]->(s)
            """, {
                "id"      : row["id_vendedor"],
                "nombre"  : row["nombre"] + " " + row["apellido"],
                "sucursal": row["sucursal"]
            })
        print("Vendedores y sucursales cargados")

        # Creo relaciones de disponibilidad entre productos y sucursales
        # drop_duplicates() evita crear la misma relación dos veces
        for _, row in df_inventario[["id_producto", "nombre_producto", "sucursal"]].drop_duplicates().iterrows():
            session.run("""
                MATCH (p:Producto {id: $id})
                MERGE (s:Sucursal {nombre: $sucursal})
                MERGE (p)-[:DISPONIBLE_EN]->(s)
            """, {
                "id"      : row["id_producto"],
                "sucursal": row["sucursal"]
            })
        print("Disponibilidad por sucursal cargada")

    print("\nGrafo completo")

cargar_grafo()

# Prueba con una consulta Cypher para verificar que el grafo se cargó bien
with driver.session() as session:
    resultado = session.run("""
        MATCH (p:Producto)-[:PERTENECE_A]->(c:Categoria {nombre: 'Cocina'})
        RETURN p.nombre AS producto, p.precio AS precio
        LIMIT 5
    """)
    print("\n=== Productos de Cocina ===")
    for r in resultado:
        print(f"  {r['producto']} - ${r['precio']}")

Grafo limpio
Productos, categorías y marcas cargados
Vendedores y sucursales cargados
Disponibilidad por sucursal cargada

Grafo completo

=== Productos de Cocina ===
  Freidora de Aire Plus - $1156.62
  Turbo Deshidratador - $2095.8
  Ultra Deshidratador - $1140.36
  Deshidratador - $297.84
  Plus Deshidratador 2024 - $2141.41


In [13]:
# CELDA 13: Generación dinámica de Cypher con LLM

def generar_cypher(consulta_usuario):
    """
    Convierte lenguaje natural a Cypher usando el LLM con few-shot prompting.
    Si el modelo no genera un Cypher válido, usa un fallback basado en palabras clave.

    """
    # Le explico al modelo la estructura del grafo y le doy ejemplos concretos
    prompt_sistema = """Sos un experto en Neo4j y Cypher. Generá SOLO la consulta Cypher, sin explicaciones ni markdown.

El grafo tiene estos nodos y relaciones:
- (:Producto {id, nombre, precio})-[:PERTENECE_A]->(:Categoria {nombre})
- (:Producto {id, nombre, precio})-[:PERTENECE_A]->(:Subcategoria {nombre})
- (:Producto {id, nombre, precio})-[:ES_DE_MARCA]->(:Marca {nombre})
- (:Vendedor {id, nombre})-[:TRABAJA_EN]->(:Sucursal {nombre})
- (:Producto {id, nombre, precio})-[:DISPONIBLE_EN]->(:Sucursal {nombre})

Categorías válidas: Cocina, Climatización, Lavado, Audio y Video

EJEMPLOS:
Pregunta: ¿Qué productos están relacionados con la categoría Cocina?
Cypher: MATCH (p:Producto)-[:PERTENECE_A]->(c:Categoria {nombre: 'Cocina'}) RETURN p.nombre AS producto, p.precio AS precio LIMIT 10;

Pregunta: ¿Qué productos son de la marca TechHome?
Cypher: MATCH (p:Producto)-[:ES_DE_MARCA]->(m:Marca {nombre: 'TechHome'}) RETURN p.nombre AS producto, p.precio AS precio LIMIT 10;

Pregunta: ¿Qué vendedores trabajan en Mendoza?
Cypher: MATCH (v:Vendedor)-[:TRABAJA_EN]->(s:Sucursal {nombre: 'Mendoza'}) RETURN v.nombre AS vendedor, s.nombre AS sucursal LIMIT 10;

Pregunta: ¿En qué sucursales está disponible la licuadora?
Cypher: MATCH (p:Producto)-[:DISPONIBLE_EN]->(s:Sucursal) WHERE LOWER(p.nombre) CONTAINS 'licuadora' RETURN p.nombre AS producto, s.nombre AS sucursal LIMIT 10;

Pregunta: ¿Qué productos hay en la subcategoría Preparación?
Cypher: MATCH (p:Producto)-[:PERTENECE_A]->(s:Subcategoria {nombre: 'Preparación'}) RETURN p.nombre AS producto, p.precio AS precio LIMIT 10;"""

    prompt_usuario = f"Pregunta: {consulta_usuario}\nCypher:"

    cypher_raw = generar(prompt_sistema, prompt_usuario, max_tokens=100)

    # Limpio el output del modelo
    cypher = re.sub(r"```cypher|```", "", cypher_raw).strip()
    cypher = cypher.strip().split("\n")[0]

    # El modelo a veces pega LIMIT sin espacio
    cypher = re.sub(r"(\w)(LIMIT)", r"\1 LIMIT", cypher)

    # Corrijo nombres de relaciones que el modelo escribe mal
    relaciones_correctas = {
        "ES_DE_MARBKA" : "ES_DE_MARCA",
        "ES_DE_MARCA_" : "ES_DE_MARCA",
        "r:ES_DE_MARCA" : "ES_DE_MARCA",
        "TRABAJA_IN"   : "TRABAJA_EN",
        "TRABAJO_CON"  : "TRABAJA_EN",
        "TRABAJA_CON"  : "TRABAJA_EN",
        "DISPONIBLE_EN_EN" : "DISPONIBLE_EN",
    }
    for mal, bien in relaciones_correctas.items():
        cypher = cypher.replace(mal, bien)

    # Si el modelo no generó RETURN, uso fallback según palabras clave
    if "RETURN" not in cypher.upper():
        consulta_lower = consulta_usuario.lower()
        if "marca" in consulta_lower:
            cypher = "MATCH (p:Producto)-[:ES_DE_MARCA]->(m:Marca) RETURN p.nombre AS producto, m.nombre AS marca LIMIT 10;"
        elif "vendedor" in consulta_lower or "sucursal" in consulta_lower:
            cypher = "MATCH (v:Vendedor)-[:TRABAJA_EN]->(s:Sucursal) RETURN v.nombre AS vendedor, s.nombre AS sucursal LIMIT 10;"
        else:
            cypher = "MATCH (p:Producto)-[:PERTENECE_A]->(c:Categoria) RETURN p.nombre AS producto, c.nombre AS categoria LIMIT 10;"

    return cypher

def buscar_grafo_llm(consulta_natural):
    """Ejecuta el Cypher generado por el LLM y devuelve los resultados."""
    cypher = generar_cypher(consulta_natural)
    try:
        with driver.session() as session:
            resultado = session.run(cypher)
            registros = [dict(r) for r in resultado]
        return {"cypher": cypher, "resultados": registros}
    except Exception as e:
        return {"cypher": cypher, "resultados": [], "error": str(e)}

# Pruebo con consultas representativas
consultas = [
    "¿Qué productos están relacionados con la categoría Cocina?",
    "¿Qué productos son de la marca TechHome?",
    "¿Qué vendedores trabajan en Mendoza?",
]

for c in consultas:
    print(f"\nConsulta: {c}")
    r = buscar_grafo_llm(c)
    print(f"Cypher generado: {r['cypher']}")
    if "error" in r:
        print(f"Error: {r['error']}")
    else:
        for item in r["resultados"][:3]:
            print(f"  {item}")


Consulta: ¿Qué productos están relacionados con la categoría Cocina?
Cypher generado: MATCH (p:Producto)-[:PERTENECE_A]->(c:Categoria {nombre: 'Cocina'}) RETURN p.nombre AS producto, c.nombre AS categoria, p.precio AS precio LIMIT 10
  {'producto': 'Freidora de Aire Plus', 'categoria': 'Cocina', 'precio': 1156.62}
  {'producto': 'Turbo Deshidratador', 'categoria': 'Cocina', 'precio': 2095.8}
  {'producto': 'Ultra Deshidratador', 'categoria': 'Cocina', 'precio': 1140.36}

Consulta: ¿Qué productos son de la marca TechHome?
Cypher generado: MATCH (p:Producto)<-[ES_DE_MARCA]-(m:Marca {nombre: 'TechHome'}) RETURN p.nombre AS producto, p.precio AS precio LIMIT 10;

Consulta: ¿Qué vendedores trabajan en Mendoza?
Cypher generado: MATCH (v:Vendedor)-[:TRABAJA_EN]->(s:Sucursal {nombre:'Mendoza'}) RETURN DISTINCT v.nombre AS vendedor, s.nombre AS sucursal LIMIT 10
  {'vendedor': 'Emanuel Navarro', 'sucursal': 'Mendoza'}
  {'vendedor': 'Laura Rodríguez', 'sucursal': 'Mendoza'}
  {'vendedor': 'Thi

In [14]:
# CELDA 14: Función de búsqueda en grafos (por palabras clave)

def buscar_grafo(consulta_natural):
    """
    Búsqueda en grafos usando patrones de palabras clave.
    Es más robusta que buscar_grafo_llm porque no depende del LLM para generar Cypher.
    Se usa como función principal en el pipeline.

    """
    consulta = consulta_natural.lower()
    resultados = []
    cypher = ""

    with driver.session() as session:

        # Detecto si pregunta por una categoría específica
        if "categoría" in consulta or "categoria" in consulta or "relacionados" in consulta:
            categorias = ["Cocina", "Climatización", "Lavado", "Audio y Video"]
            # Busco si alguna categoría válida aparece en la consulta
            categoria_encontrada = next((c for c in categorias if c.lower() in consulta), "Cocina")
            cypher = """
                MATCH (p:Producto)-[:PERTENECE_A]->(c:Categoria {nombre: $categoria})
                RETURN p.nombre AS resultado, c.nombre AS relacion, p.precio AS precio
                LIMIT 10
            """
            res = session.run(cypher, {"categoria": categoria_encontrada})
            resultados = [dict(r) for r in res]

        # Detecto si pregunta por una marca específica
        elif "marca" in consulta:
            marcas = df_productos["marca"].unique().tolist()
            marca_encontrada = next((m for m in marcas if m.lower() in consulta), None)
            if marca_encontrada:
                cypher = """
                    MATCH (p:Producto)-[:ES_DE_MARCA]->(m:Marca {nombre: $marca})
                    RETURN p.nombre AS resultado, m.nombre AS relacion, p.precio AS precio
                    LIMIT 10
                """
                res = session.run(cypher, {"marca": marca_encontrada})
                resultados = [dict(r) for r in res]

        # Detecto si pregunta por disponibilidad en sucursal
        elif "sucursal" in consulta or "disponible" in consulta:
            sucursales = df_vendedores["sucursal"].unique().tolist()
            sucursal_encontrada = next((s for s in sucursales if s.lower() in consulta), None)
            if sucursal_encontrada:
                cypher = """
                    MATCH (p:Producto)-[:DISPONIBLE_EN]->(s:Sucursal {nombre: $sucursal})
                    RETURN p.nombre AS resultado, s.nombre AS relacion, p.precio AS precio
                    LIMIT 10
                """
                res = session.run(cypher, {"sucursal": sucursal_encontrada})
                resultados = [dict(r) for r in res]

        # Detecto si pregunta por vendedores
        elif "vendedor" in consulta or "vendedores" in consulta:
            sucursales = df_vendedores["sucursal"].unique().tolist()
            sucursal_encontrada = next((s for s in sucursales if s.lower() in consulta), None)
            if sucursal_encontrada:
                cypher = """
                    MATCH (v:Vendedor)-[:TRABAJA_EN]->(s:Sucursal {nombre: $sucursal})
                    RETURN v.nombre AS resultado, s.nombre AS relacion
                    LIMIT 10
                """
                res = session.run(cypher, {"sucursal": sucursal_encontrada})
                resultados = [dict(r) for r in res]

    return {
        "cypher"    : cypher.strip(),
        "resultados": resultados
    }

# Pruebo con consultas representativas
print("=== Productos relacionados con Cocina ===")
r = buscar_grafo("¿Qué productos están relacionados con la categoría Cocina?")
print(f"Cypher: {r['cypher'][:100]}...")
for item in r["resultados"][:3]:
    print(f"  {item}")

print("\n=== Productos de marca TechHome ===")
r = buscar_grafo("¿Qué productos son de la marca TechHome?")
for item in r["resultados"][:3]:
    print(f"  {item}")

=== Productos relacionados con Cocina ===
Cypher: MATCH (p:Producto)-[:PERTENECE_A]->(c:Categoria {nombre: $categoria})
                RETURN p.nombr...
  {'resultado': 'Freidora de Aire Plus', 'relacion': 'Cocina', 'precio': 1156.62}
  {'resultado': 'Turbo Deshidratador', 'relacion': 'Cocina', 'precio': 2095.8}
  {'resultado': 'Ultra Deshidratador', 'relacion': 'Cocina', 'precio': 1140.36}

=== Productos de marca TechHome ===
  {'resultado': 'Turbo Deshidratador', 'relacion': 'TechHome', 'precio': 2095.8}
  {'resultado': 'Yogurtera', 'relacion': 'TechHome', 'precio': 2623.0}
  {'resultado': 'Exprimidor', 'relacion': 'TechHome', 'precio': 1942.87}


### 1.5 Clasificador de Intención

Determina a qué base de datos enviar cada consulta:
- **Vectorial** → preguntas sobre uso, instrucciones, reseñas u opiniones
- **Tabular** → preguntas sobre precios, stock, ventas o devoluciones
- **Grafos** → preguntas sobre relaciones entre categorías, marcas o sucursales

Se implementaron y compararon dos enfoques: clasificador entrenado (TF-IDF + Regresión Logística) y clasificador Few-Shot con LLM.

In [15]:
# CELDA 15: Clasificador de Intención (entrenado)

# Datos de entrenamiento: Preguntas sintéticas etiquetadas por fuente
datos_entrenamiento = [
    # VECTORIAL: preguntas sobre uso, descripción, reseñas, manuales
    ("¿Cómo uso mi licuadora para hacer smoothies?", "vectorial"),
    ("¿Cómo funciona la freidora de aire?", "vectorial"),
    ("¿Qué opinan los usuarios de esta cafetera?", "vectorial"),
    ("Quiero una licuadora con buenas reseñas", "vectorial"),
    ("¿Cuáles son las instrucciones de uso del microondas?", "vectorial"),
    ("¿Qué dicen los clientes sobre la plancha?", "vectorial"),
    ("¿Cómo limpio mi cafetera?", "vectorial"),
    ("¿Para qué sirve el modo pulse de la licuadora?", "vectorial"),
    ("Busco productos bien valorados por los usuarios", "vectorial"),
    ("¿Cómo se instala el horno eléctrico?", "vectorial"),
    ("¿Qué mantenimiento necesita la heladera?", "vectorial"),
    ("Leí que la sandwichera tiene problemas, es verdad?", "vectorial"),
    ("¿Cuál es la experiencia de uso del aire acondicionado?", "vectorial"),
    ("¿Cómo hago pan con la panificadora?", "vectorial"),
    ("¿Qué voltaje requiere el rallador eléctrico?", "vectorial"),

    # TABULAR: preguntas sobre precios, stock, ventas, devoluciones
    ("¿Cuáles son las licuadoras de menos de $200?", "tabular"),
    ("¿Qué productos tienen stock disponible?", "tabular"),
    ("¿Cuánto cuesta la cafetera?", "tabular"),
    ("¿Cuáles son los productos más vendidos?", "tabular"),
    ("¿Qué devoluciones hubo este mes?", "tabular"),
    ("¿Cuántas unidades quedan de la freidora?", "tabular"),
    ("¿Cuál es el precio mínimo de una heladera?", "tabular"),
    ("¿Qué productos cuestan menos de $500?", "tabular"),
    ("¿Cuántas ventas se hicieron en marzo?", "tabular"),
    ("¿Qué productos tienen descuento?", "tabular"),
    ("¿Cuál es el producto más caro del catálogo?", "tabular"),
    ("¿Cuántas devoluciones fueron aprobadas?", "tabular"),
    ("Dame los productos con más de 100 unidades en stock", "tabular"),
    ("¿Qué métodos de pago se usaron más?", "tabular"),
    ("¿Cuánto se vendió en total el año pasado?", "tabular"),
    ("¿Qué licuadoras tienen disponibles?", "tabular"),
    ("¿Qué heladeras hay disponibles?", "tabular"),
    ("¿Qué productos están disponibles?", "tabular"),
    ("¿Qué electrodomésticos tienen en stock?", "tabular"),

    # GRAFOS: preguntas sobre relaciones entre productos, categorías, marcas
    ("¿Qué productos están relacionados con la categoría Cocina?", "grafos"),
    ("¿Qué marcas tienen productos en Climatización?", "grafos"),
    ("¿Qué productos vende la sucursal de Mendoza?", "grafos"),
    ("¿Quiénes son los vendedores de Buenos Aires?", "grafos"),
    ("¿Qué categorías existen en el catálogo?", "grafos"),
    ("¿Qué productos de TechHome hay disponibles?", "grafos"),
    ("¿En qué sucursales está disponible la licuadora?", "grafos"),
    ("¿Qué subcategorías tiene Lavado?", "grafos"),
    ("¿Qué productos están relacionados con Refrigeración?", "grafos"),
    ("¿Qué vendedores trabajan en Córdoba?", "grafos"),
    ("¿Qué marcas operan en la categoría Audio y Video?", "grafos"),
    ("¿En qué sucursales trabaja Juan?", "grafos"),
    ("¿Qué productos de ChefMaster están en Cocina?", "grafos"),
    ("¿Cuáles son las sucursales disponibles?", "grafos"),
    ("¿Qué productos tiene la subcategoría Preparación?", "grafos"),
]

# Separar textos y etiquetas
textos    = [d[0] for d in datos_entrenamiento]
etiquetas = [d[1] for d in datos_entrenamiento]

# Entreno clasificador
clasificador = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf",   LogisticRegression(max_iter=1000))
])

clasificador.fit(textos, etiquetas)
print("Clasificador entrenado")

# Función de clasificación
def clasificar_intencion(consulta):
    """Devuelve la fuente de datos más adecuada para la consulta."""
    return clasificador.predict([consulta])[0]

# Pruebas
pruebas = [
    "¿Cómo uso mi licuadora para hacer smoothies?",
    "¿Cuáles son las licuadoras de menos de $200?",
    "¿Qué opinan los usuarios de esta cafetera?",
    "Quiero una licuadora con buenas reseñas",
    "¿Qué productos están relacionados con la categoría Cocina?",
]

print("\n=== PRUEBAS DEL CLASIFICADOR ===")
for p in pruebas:
    print(f"  '{p[:50]}...' → {clasificar_intencion(p)}")

Clasificador entrenado

=== PRUEBAS DEL CLASIFICADOR ===
  '¿Cómo uso mi licuadora para hacer smoothies?...' → vectorial
  '¿Cuáles son las licuadoras de menos de $200?...' → tabular
  '¿Qué opinan los usuarios de esta cafetera?...' → vectorial
  'Quiero una licuadora con buenas reseñas...' → vectorial
  '¿Qué productos están relacionados con la categoría...' → grafos


In [16]:
# CELDA 16: Clasificador Few-Shot con LLM

def clasificar_intencion_llm(consulta):

    """

    Clasifica la intención usando el LLM con Few-Shot prompting.

    Devuelve: 'vectorial', 'tabular' o 'grafos'

    """

    prompt_sistema = """Tu tarea es clasificar preguntas en una de estas tres categorías:

- vectorial: preguntas sobre uso, instrucciones, reseñas, opiniones, manuales, descripción de productos

- tabular: preguntas sobre precios, stock, cantidades, ventas, devoluciones, métodos de pago

- grafos: preguntas sobre relaciones entre productos, categorías, marcas, sucursales, vendedores

Respondé ÚNICAMENTE con una palabra: vectorial, tabular o grafos. Sin explicaciones.

EJEMPLOS:

Pregunta: ¿Cómo uso mi licuadora para hacer smoothies?

Categoría: vectorial

Pregunta: ¿Cuáles son las licuadoras de menos de $200?

Categoría: tabular

Pregunta: ¿Qué opinan los usuarios de esta cafetera?

Categoría: vectorial

Pregunta: ¿Qué productos están relacionados con la categoría Cocina?

Categoría: grafos

Pregunta: ¿Cuánto cuesta la heladera?

Categoría: tabular

Pregunta: ¿Qué vendedores trabajan en Mendoza?

Categoría: grafos

Pregunta: ¿Cómo limpio mi cafetera?

Categoría: vectorial

Pregunta: ¿Cuántas devoluciones hubo este mes?

Categoría: tabular"""

    prompt_usuario = f"Pregunta: {consulta}\nCategoría:"

    respuesta = generar(prompt_sistema, prompt_usuario, max_tokens=10)

    # Limpio y valido respuesta

    respuesta = respuesta.strip().lower().split()[0]

    if respuesta not in ["vectorial", "tabular", "grafos"]:
        respuesta = "vectorial"  # default
    return respuesta

# Pruebas con los 5 prompts obligatorios

prompts = [

    "¿Cómo uso mi licuadora para hacer smoothies?",

    "¿Cuáles son las licuadoras de menos de $200?",

    "¿Qué opinan los usuarios de esta cafetera?",

    "Quiero una licuadora con buenas reseñas",

    "¿Qué productos están relacionados con la categoría Cocina?",

]

print("=== CLASIFICADOR FEW-SHOT (LLM) ===")

for p in prompts:
    intencion = clasificar_intencion_llm(p)
    print(f"  '{p[:50]}' → {intencion}")

=== CLASIFICADOR FEW-SHOT (LLM) ===
  '¿Cómo uso mi licuadora para hacer smoothies?' → vectorial
  '¿Cuáles son las licuadoras de menos de $200?' → tabular
  '¿Qué opinan los usuarios de esta cafetera?' → vectorial
  'Quiero una licuadora con buenas reseñas' → vectorial
  '¿Qué productos están relacionados con la categoría' → grafos


#### Comparación de Clasificadores

Se comparan ambos enfoques usando un conjunto de prueba de 15 preguntas no vistas durante el entrenamiento.

In [17]:
# CELDA 17: Comparación de clasificadores


from sklearn.metrics import classification_report, accuracy_score

# Dataset de prueba (preguntas que NO se usaron para entrenar)
datos_test = [
    ("¿Cómo preparo jugo con la licuadora?", "vectorial"),
    ("¿Qué dicen los clientes sobre el microondas?", "vectorial"),
    ("¿Cuál es el manual de la freidora?", "vectorial"),
    ("¿Qué voltaje necesita la cafetera?", "vectorial"),
    ("¿Tiene buenas reseñas la plancha?", "vectorial"),
    ("¿Cuál es el precio de la heladera?", "tabular"),
    ("¿Cuántas unidades hay en stock?", "tabular"),
    ("¿Qué productos cuestan menos de $300?", "tabular"),
    ("¿Cuántas ventas hubo en enero?", "tabular"),
    ("¿Qué métodos de pago se usaron?", "tabular"),
    ("¿Qué marcas hay en la categoría Lavado?", "grafos"),
    ("¿En qué sucursales está disponible el freezer?", "grafos"),
    ("¿Qué productos tiene ChefMaster?", "grafos"),
    ("¿Quiénes son los vendedores de Córdoba?", "grafos"),
    ("¿Qué subcategorías tiene Climatización?", "grafos"),
]

textos_test   = [d[0] for d in datos_test]
etiquetas_test = [d[1] for d in datos_test]

# Predicciones clasificador entrenado
pred_entrenado = [clasificar_intencion(t) for t in textos_test]

# Predicciones clasificador LLM
print("Clasificando con LLM ...")
pred_llm = [clasificar_intencion_llm(t) for t in textos_test]

# Métricas
print("\n=== CLASIFICADOR ENTRENADO (Logistic Regression) ===")
print(f"Accuracy: {accuracy_score(etiquetas_test, pred_entrenado):.2f}")
print(classification_report(etiquetas_test, pred_entrenado))

print("\n=== CLASIFICADOR FEW-SHOT (LLM) ===")
print(f"Accuracy: {accuracy_score(etiquetas_test, pred_llm):.2f}")
print(classification_report(etiquetas_test, pred_llm))

Clasificando con LLM ...

=== CLASIFICADOR ENTRENADO (Logistic Regression) ===
Accuracy: 0.93
              precision    recall  f1-score   support

      grafos       1.00      1.00      1.00         5
     tabular       0.83      1.00      0.91         5
   vectorial       1.00      0.80      0.89         5

    accuracy                           0.93        15
   macro avg       0.94      0.93      0.93        15
weighted avg       0.94      0.93      0.93        15


=== CLASIFICADOR FEW-SHOT (LLM) ===
Accuracy: 1.00
              precision    recall  f1-score   support

      grafos       1.00      1.00      1.00         5
     tabular       1.00      1.00      1.00         5
   vectorial       1.00      1.00      1.00         5

    accuracy                           1.00        15
   macro avg       1.00      1.00      1.00        15
weighted avg       1.00      1.00      1.00        15



### 1.6 Pipeline de Recuperación

Búsqueda híbrida: semántica (ChromaDB) + BM25, combinadas con ReRank.

El score final se calcula como media ponderada: 50% semántico + 50% BM25. Los resultados se ordenan por score final y se devuelven los K mejores.

Para grafos y datos tabulares se generan consultas SQL/Cypher dinámicamente — no se pasa la tabla o el grafo completo al contexto.

In [18]:
# CELDA 18: Búsqueda híbrida (BM25 + semántica) + ReRank

# Preparar corpus para BM25
print("Preparando corpus BM25...")

# Recolectar todos los documentos indexados en ChromaDB
todos_los_docs = collection.get()
corpus_textos = todos_los_docs["documents"]
corpus_ids = todos_los_docs["ids"]
corpus_metas = todos_los_docs["metadatas"]

# Tokenización
corpus_tokenizado = [doc.lower().split() for doc in corpus_textos]

# Crear índice BM25
bm25 = BM25Okapi(corpus_tokenizado)
print(f"BM25 listo con {len(corpus_textos)} documentos")

# Función de búsqueda híbrida
def buscar_hibrido(consulta, k=5, filtro_fuente=None):
    """
    Combina búsqueda semántica (ChromaDB) + BM25 y reordena por score combinado.

    """
    # Búsqueda semántica
    resultados_semanticos = buscar_vectorial(consulta, k=k*2, filtro_fuente=filtro_fuente)

    # Búsqueda BM25
    tokens_consulta = consulta.lower().split()
    scores_bm25 = bm25.get_scores(tokens_consulta)

    # Armar diccionario de scores BM25 por índice
    bm25_top_indices = scores_bm25.argsort()[::-1][:k*2]

    resultados_bm25 = []
    for idx in bm25_top_indices:
        # Filtrar por fuente si se especificó
        if filtro_fuente and corpus_metas[idx].get("fuente") != filtro_fuente:
            continue
        resultados_bm25.append({
            "contenido" : corpus_textos[idx],
            "metadata"  : corpus_metas[idx],
            "score_bm25": float(scores_bm25[idx])
        })

    # Combinar resultados
    # Normalizar scores BM25 entre 0 y 1
    max_bm25 = max([r["score_bm25"] for r in resultados_bm25], default=1)
    for r in resultados_bm25:
        r["score_bm25"] = r["score_bm25"] / max_bm25 if max_bm25 > 0 else 0

    # Unir resultados semánticos y BM25
    combinados = {}
    for r in resultados_semanticos:
        key = r["contenido"][:50]
        combinados[key] = {
            "contenido"     : r["contenido"],
            "metadata"      : r["metadata"],
            "score_semantico": r["score"],
            "score_bm25"    : 0.0
        }

    for r in resultados_bm25:
        key = r["contenido"][:50]
        if key in combinados:
            combinados[key]["score_bm25"] = r["score_bm25"]
        else:
            combinados[key] = {
                "contenido"     : r["contenido"],
                "metadata"      : r["metadata"],
                "score_semantico": 0.0,
                "score_bm25"    : r["score_bm25"]
            }

    # ReRank: score final = promedio ponderado
    # 50% semántico + 50% BM25
    for key in combinados:
        combinados[key]["score_final"] = (
            0.5 * combinados[key]["score_semantico"] +
            0.5 * combinados[key]["score_bm25"]
        )

    # Ordenar por score final y devolver top K
    resultados_finales = sorted(combinados.values(), key=lambda x: x["score_final"], reverse=True)[:k]

    return resultados_finales

# Prueba
print("\n=== BÚSQUEDA HÍBRIDA: licuadora smoothies ===")
resultados = buscar_hibrido("¿Cómo uso mi licuadora para hacer smoothies?", k=3)
for r in resultados:
    print(f"  Score final: {r['score_final']:.4f} (sem: {r['score_semantico']:.4f} | bm25: {r['score_bm25']:.4f})")
    print(f"  Contenido: {r['contenido'][:150]}...")
    print()

Preparando corpus BM25...
BM25 listo con 9456 documentos

=== BÚSQUEDA HÍBRIDA: licuadora smoothies ===
  Score final: 0.5000 (sem: 0.0000 | bm25: 1.0000)
  Contenido: ¿Cómo se usa correctamente este producto? El Licuadora de TechHome está diseñado para uso doméstico. Revise el manual del producto (código P0001) para...

  Score final: 0.4947 (sem: 0.0000 | bm25: 0.9895)
  Contenido: ¿Cómo extiendo la vida útil del producto? El Licuadora de HomeChef está diseñado para uso doméstico. Revise el manual del producto (código P0005) para...

  Score final: 0.2712 (sem: 0.5424 | bm25: 0.0000)
  Contenido: ¿Cuál es la forma correcta de encenderlo? El Mixer de KitchenPro está diseñado para uso doméstico. Revise el manual del producto (código P0025) para m...



### 1.7 Checkpoint de Recuperación

Punto de control del pipeline de recuperación completo, ejecutado de forma aislada sin generar respuesta con el LLM.

Para cada uno de los 5 prompts obligatorios se muestra:
- Intención detectada y fuente de datos seleccionada
- Consulta generada (SQL o Cypher) para fuentes tabular y de grafos
- Fragmentos o registros recuperados con contenido, fuente, metadata y puntaje

In [19]:
# CELDA 19: Checkpoint de Recuperación

print("=" * 60)
print("        CHECKPOINT DE RECUPERACIÓN — ITEM 1.7")
print("=" * 60)

prompts_obligatorios = [
    "¿Cómo uso mi licuadora para hacer smoothies?",
    "¿Cuáles son las licuadoras de menos de $200?",
    "¿Qué opinan los usuarios de esta cafetera?",
    "Quiero una licuadora con buenas reseñas",
    "¿Qué productos están relacionados con la categoría Cocina?",
]

for i, prompt in enumerate(prompts_obligatorios, 1):
    print(f"\n{'='*60}")
    print(f"PROMPT {i}: {prompt}")
    print(f"{'='*60}")

    # Clasificar intención
    intencion = clasificar_intencion(prompt)
    print(f"\nIntención detectada: {intencion.upper()}")

    # Recuperar según la fuente
    if intencion == "vectorial":
        print(f"Fuente seleccionada: BASE VECTORIAL")
        print(f"Método: Búsqueda híbrida (semántica + BM25) + ReRank")

        # Si pregunta por opiniones, filtrar solo reseñas
        prompt_lower = prompt.lower()
        if any(p in prompt_lower for p in ["opinan", "opinión", "reseña", "experiencia"]):
            resultados = buscar_hibrido(prompt, k=3, filtro_fuente="resena")
        else:
            resultados = buscar_hibrido(prompt, k=3)

        print(f"\n   Fragmentos recuperados:")
        for j, r in enumerate(resultados, 1):
            print(f"\n   [{j}] Score final: {r['score_final']:.4f} | sem: {r['score_semantico']:.4f} | bm25: {r['score_bm25']:.4f}")
            print(f"       Fuente:    {r['metadata'].get('fuente', 'N/A')}")
            print(f"       Metadata:  {r['metadata']}")
            print(f"       Contenido: {r['contenido'][:200]}...")

    elif intencion == "tabular":
        print(f"Fuente seleccionada: BASE TABULAR")
        sql = generar_sql(prompt)
        print(f"\n   Consulta SQL generada:\n   {sql}")
        resultado = buscar_tabular(sql)
        print(f"\n   Registros recuperados:")
        if isinstance(resultado, pd.DataFrame):
            print(resultado.head(5).to_string(index=False))
        else:
            print(f"   {resultado}")

    elif intencion == "grafos":
        print(f"Fuente seleccionada: BASE DE GRAFOS")
        resultado = buscar_grafo(prompt)
        print(f"\n   Consulta Cypher generada:\n   {resultado['cypher']}")
        print(f"\n   Registros recuperados:")
        if "error" in resultado:
            print(f"   Error: {resultado['error']}")
        else:
            for r in resultado["resultados"][:5]:
                print(f"   {r}")

print(f"\n{'='*60}")
print("        CHECKPOINT COMPLETADO")
print(f"{'='*60}")

        CHECKPOINT DE RECUPERACIÓN — ITEM 1.7

PROMPT 1: ¿Cómo uso mi licuadora para hacer smoothies?

Intención detectada: VECTORIAL
Fuente seleccionada: BASE VECTORIAL
Método: Búsqueda híbrida (semántica + BM25) + ReRank

   Fragmentos recuperados:

   [1] Score final: 0.5000 | sem: 0.0000 | bm25: 1.0000
       Fuente:    faq
       Metadata:  {'fuente': 'faq', 'id_faq': 'FAQ00004', 'nombre_producto': 'Licuadora', 'categoria': 'Uso', 'id_producto': 'P0001'}
       Contenido: ¿Cómo se usa correctamente este producto? El Licuadora de TechHome está diseñado para uso doméstico. Revise el manual del producto (código P0001) para más detalles. Ante cualquier duda, contacte a nue...

   [2] Score final: 0.4947 | sem: 0.0000 | bm25: 0.9895
       Fuente:    faq
       Metadata:  {'id_producto': 'P0005', 'id_faq': 'FAQ00048', 'categoria': 'Mantenimiento', 'fuente': 'faq', 'nombre_producto': 'Licuadora'}
       Contenido: ¿Cómo extiendo la vida útil del producto? El Licuadora de HomeChef está d

### 1.9 Generación de Respuestas y Conversación con Memoria

Integra todos los componentes en un chatbot conversacional con historial de turnos.

La función `enriquecer_consulta()` detecta preguntas ambiguas ("esa", "la más barata", "¿y cuál?") y agrega el contexto del turno anterior para mejorar la búsqueda.

Flujo de cada turno: clasificar intención → recuperar contexto → generar respuesta con el LLM usando solo la información recuperada.

In [20]:
# CELDA 20: Generación de respuestas + memoria

historial = []

def resetear_historial():
    global historial
    historial = []
    print("Historial reseteado")

def enriquecer_consulta(consulta_usuario):
    """Agrega contexto del historial a la consulta para mejorar el SQL/búsqueda."""
    if not historial:
        return consulta_usuario

    # Si la consulta es corta y ambigua, agregarle contexto del turno anterior
    consulta_lower = consulta_usuario.lower()
    palabras_ambiguas = ["cuál", "esa", "ese", "eso", "la más", "el más", "cuánto", "y cuál"]

    if any(p in consulta_lower for p in palabras_ambiguas):
        ultimo_turno = historial[-1]["usuario"]
        return f"{consulta_usuario} (contexto: {ultimo_turno})"

    return consulta_usuario

def chatbot(consulta_usuario, verbose=True):
    global historial

    # Enriquecer consulta con contexto del historial
    consulta_enriquecida = enriquecer_consulta(consulta_usuario)

    # Paso 1: Clasificar intención
    intencion = clasificar_intencion(consulta_enriquecida)

    # Paso 2: Recuperar contexto
    contexto = ""

    if intencion == "vectorial":
        consulta_lower = consulta_usuario.lower()
        if any(p in consulta_lower for p in ["opinan", "opinión", "reseña", "experiencia", "usuarios dicen"]):
            resultados = buscar_hibrido(consulta_enriquecida, k=3, filtro_fuente="resena")
        else:
            resultados = buscar_hibrido(consulta_enriquecida, k=3)
        fragmentos = [r["contenido"] for r in resultados]
        contexto = "\n\n".join(fragmentos)

    elif intencion == "tabular":
        sql = generar_sql(consulta_enriquecida)
        resultado = buscar_tabular(sql)
        if isinstance(resultado, pd.DataFrame) and not resultado.empty:
            contexto = resultado.head(5).to_string(index=False)
        else:
            contexto = "No se encontraron productos que cumplan esa condición en nuestra base de datos."

    elif intencion == "grafos":
        resultado = buscar_grafo(consulta_enriquecida)
        if resultado["resultados"]:
            contexto = "\n".join([str(r) for r in resultado["resultados"]])
        else:
            contexto = "No se encontraron relaciones para esa consulta."

    if verbose:
        print(f"   [Intención: {intencion}]")
        print(f"   [Consulta enriquecida: {consulta_enriquecida}]")
        print(f"   [Contexto: {contexto[:200]}...]")

    # Paso 3: Generar respuesta
    prompt_sistema = """Sos un asistente de ElectroHogar Argentina. Respondé SOLO con la información del CONTEXTO. Sé MUY breve, máximo 2 oraciones. No inventes nada."""

    prompt_usuario = f"""CONTEXTO: {contexto}

PREGUNTA: {consulta_usuario}

RESPUESTA (máximo 2 oraciones, solo con info del contexto):"""

    respuesta = generar(prompt_sistema, prompt_usuario, max_tokens=80)

    # Limpiar si el modelo repite el prompt
    for tag in ["CONTEXTO:", "PREGUNTA:", "RESPUESTA:"]:
        if tag in respuesta:
            respuesta = respuesta.split(tag)[0].strip()

    # Cortar en el segundo punto si es muy larga
    oraciones = respuesta.split(".")
    if len(oraciones) > 3:
        respuesta = ".".join(oraciones[:2]) + "."

    historial.append({
        "usuario"  : consulta_usuario,
        "asistente": respuesta
    })

    return respuesta

# Pruebas
resetear_historial()

preguntas = [
    "¿Qué licuadoras tienen disponibles?",
    "¿Y cuál es la más barata de todo el catálogo?",
    "¿Qué opinan los usuarios de la licuadora?",
    "¿Qué productos hay en la categoría Cocina?",
]

for p in preguntas:
    print(f"\n USUARIO: {p}")
    respuesta = chatbot(p)
    print(f"ASISTENTE: {respuesta}")
    print()

Historial reseteado

 USUARIO: ¿Qué licuadoras tienen disponibles?
   [Intención: tabular]
   [Consulta enriquecida: ¿Qué licuadoras tienen disponibles?]
   [Contexto:             nombre  precio_usd  stock
         Licuadora      283.63    108
         Licuadora     1273.06    114
Plus Licuadora Pro      329.07     97
Compacto Licuadora      259.42     75
         L...]
ASISTENTE: Licuadora - $283.63 disponible en cantidad 108  
Licuadora - $1273.


 USUARIO: ¿Y cuál es la más barata de todo el catálogo?
   [Intención: tabular]
   [Consulta enriquecida: ¿Y cuál es la más barata de todo el catálogo? (contexto: ¿Qué licuadoras tienen disponibles?)]
   [Contexto:             nombre  precio_usd  stock
Compacto Licuadora      259.42     75...]
ASISTENTE: La Compacto Licuadora tiene un precio de $259.42 y está en existencia disponible.


 USUARIO: ¿Qué opinan los usuarios de la licuadora?
   [Intención: vectorial]
   [Consulta enriquecida: ¿Qué opinan los usuarios de la licuadora?]
   [Conte

## Ejercicio 2 — Agente ReAct

El sistema RAG se transforma en un agente ReAct implementado con LangChain (sin LangGraph), con 4 herramientas especializadas:

- `doc_search` → búsqueda híbrida en documentos, manuales, FAQs y reseñas
- `table_search` → consultas SQL dinámicas a productos, precios y ventas
- `graph_search` → consultas Cypher al grafo de categorías, marcas y sucursales
- `analytics_tool` → análisis estadísticos y generación de gráficos con matplotlib

El agente implementa el ciclo Thought → Action → Observation hasta generar una Final Answer.

In [21]:
# CELDA 21: Instalación para Agente ReAct

!pip install -q langchain-community matplotlib

print("Librerías instaladas")

Librerías instaladas


In [22]:
# CELDA 22: Definición de Herramientas y Agente ReAct


from langchain_core.tools import tool
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

# Tool 1: doc_search
@tool
def doc_search(consulta: str) -> str:
    """Busca en documentos, manuales, FAQs y reseñas de usuarios usando búsqueda híbrida semántica."""
    resultados = buscar_hibrido(consulta, k=3)
    if not resultados:
        return "No se encontraron documentos relevantes."
    salida = []
    for r in resultados:
        salida.append(f"[Score: {r['score_final']:.2f}] {r['contenido'][:200]}")
    return "\n---\n".join(salida)

# Tool 2: table_search
@tool
def table_search(consulta: str) -> str:
    """Busca en la base de datos tabular de productos, precios, stock, ventas y devoluciones."""
    sql = generar_sql(consulta)
    resultado = buscar_tabular(sql)
    if isinstance(resultado, pd.DataFrame) and not resultado.empty:
        return f"SQL ejecutado: {sql}\nResultados:\n{resultado.head(5).to_string(index=False)}"
    return "No se encontraron registros que cumplan esa condición."

# Tool 3: graph_search
@tool
def graph_search(consulta: str) -> str:
    """Busca relaciones en el grafo de conocimiento: categorías, marcas y sucursales."""
    resultado = buscar_grafo(consulta)
    if resultado["resultados"]:
        registros = "\n".join([str(r) for r in resultado["resultados"][:5]])
        return f"Resultados del grafo:\n{registros}"
    return "No se encontraron relaciones para esa consulta."

# Tool 4: analytics_tool
@tool
def analytics_tool(consulta: str) -> str:
    """Genera análisis estadísticos y gráficos sobre ventas, métodos de pago y devoluciones."""
    consulta_lower = consulta.lower()

    if "pago" in consulta_lower or "método" in consulta_lower:
        sql = "SELECT metodo_pago, COUNT(*) as cantidad FROM ventas GROUP BY metodo_pago ORDER BY cantidad DESC LIMIT 8;"
        df = buscar_tabular(sql)
        if isinstance(df, pd.DataFrame) and not df.empty:
            plt.figure(figsize=(8, 5))
            plt.bar(df["metodo_pago"], df["cantidad"], color="steelblue")
            plt.title("Distribución de Métodos de Pago")
            plt.xticks(rotation=45, ha="right")
            plt.tight_layout()
            plt.savefig("/content/grafico_pagos.png")
            plt.close()
            return f"Gráfico generado en /content/grafico_pagos.png\nDatos:\n{df.to_string(index=False)}"

    elif "venta" in consulta_lower or "sucursal" in consulta_lower:
        sql = "SELECT sucursal, SUM(total) as total_ventas FROM ventas GROUP BY sucursal ORDER BY total_ventas DESC LIMIT 5;"
        df = buscar_tabular(sql)
        if isinstance(df, pd.DataFrame) and not df.empty:
            plt.figure(figsize=(8, 5))
            plt.bar(df["sucursal"], df["total_ventas"], color="coral")
            plt.title("Ventas por Sucursal")
            plt.xticks(rotation=45, ha="right")
            plt.tight_layout()
            plt.savefig("/content/grafico_ventas.png")
            plt.close()
            return f"Gráfico generado en /content/grafico_ventas.png\nDatos:\n{df.to_string(index=False)}"

    elif "devolucion" in consulta_lower or "devolución" in consulta_lower:
        sql = "SELECT motivo, COUNT(*) as cantidad FROM devoluciones GROUP BY motivo ORDER BY cantidad DESC LIMIT 5;"
        df = buscar_tabular(sql)
        if isinstance(df, pd.DataFrame) and not df.empty:
            return f"Motivos de devolución:\n{df.to_string(index=False)}"

    return "Especificá si querés ver análisis de pagos, ventas por sucursal o devoluciones."

tools = [doc_search, table_search, graph_search, analytics_tool]

print("Herramientas definidas:")
for t in tools:
    print(f"   - {t.name}: {t.description[:70]}...")

Herramientas definidas:
   - doc_search: Busca en documentos, manuales, FAQs y reseñas de usuarios usando búsqu...
   - table_search: Busca en la base de datos tabular de productos, precios, stock, ventas...
   - graph_search: Busca relaciones en el grafo de conocimiento: categorías, marcas y suc...
   - analytics_tool: Genera análisis estadísticos y gráficos sobre ventas, métodos de pago ...


In [23]:
# CELDA 23: Agente ReAct

tools_dict = {t.name: t for t in tools}
historial_agente = []

def agente_react(consulta_usuario, max_iteraciones=5):
    """
    Agente ReAct que razona y actúa para responder consultas complejas.
    Implementa el ciclo: Thought → Action → Observation → Respuesta Final

    """
    print(f"\n{'='*60}")
    print(f"CONSULTA: {consulta_usuario}")
    print(f"{'='*60}")

    # Descripción de herramientas disponibles para el agente
    tools_desc = """- doc_search: buscar información sobre USO de productos, manuales técnicos, FAQs y RESEÑAS de usuarios
- table_search: consultar PRECIOS, STOCK disponible, ventas y devoluciones
- graph_search: consultar CATEGORÍAS de productos, marcas y sucursales
- analytics_tool: generar GRÁFICOS y estadísticas de métodos de pago, ventas por sucursal y devoluciones"""

    contexto_acumulado = ""
    iteracion = 0

    while iteracion < max_iteraciones:
        iteracion += 1

        # Prompt del agente con formato ReAct obligatorio
        prompt_sistema = f"""Sos un agente de ElectroHogar Argentina con acceso a estas herramientas:
{tools_desc}

Formato OBLIGATORIO para cada respuesta:
Thought: [razonamiento breve]
Action: [una de: doc_search, table_search, graph_search, analytics_tool]
Action Input: [consulta en texto simple, sin JSON]

O si ya tenés suficiente información:
Thought: [razonamiento]
Final Answer: [respuesta al usuario]

IMPORTANTE:
- Para preguntas sobre categorías → usá graph_search
- Para preguntas sobre precios/stock → usá table_search
- Para preguntas sobre uso/reseñas → usá doc_search
- Para estadísticas/gráficos/pagos → usá analytics_tool
- Action Input debe ser texto simple, nunca JSON
- Cuando tengas la respuesta, usá Final Answer inmediatamente"""

        prompt_usuario = f"""Contexto recopilado:
{contexto_acumulado if contexto_acumulado else "Ninguno aún."}

Pregunta: {consulta_usuario}
Iteración {iteracion}/{max_iteraciones}:"""

        respuesta_llm = generar(prompt_sistema, prompt_usuario, max_tokens=150)
        print(f"\n Iter {iteracion}:\n{respuesta_llm}")

        # Parsear Final Answer
        if "Final Answer:" in respuesta_llm:
            respuesta_final = respuesta_llm.split("Final Answer:")[-1].strip()
            respuesta_final = respuesta_final.split("\n\n")[0].strip()
            print(f"\n RESPUESTA FINAL: {respuesta_final}")
            historial_agente.append({
                "usuario": consulta_usuario,
                "respuesta": respuesta_final
            })
            return respuesta_final

        # Parsear Action y Action Input
        if "Action:" in respuesta_llm:
            try:
                action = respuesta_llm.split("Action:")[-1].split("\n")[0].strip()

                # Intentar parsear Action Input con mayúscula y minúscula
                if "Action Input:" in respuesta_llm:
                    action_input = respuesta_llm.split("Action Input:")[-1].split("\n")[0].strip()
                elif "Action input:" in respuesta_llm:
                    action_input = respuesta_llm.split("Action input:")[-1].split("\n")[0].strip()
                else:
                    action_input = consulta_usuario

                # Limpiar y corregir nombre de herramienta
                action = action.lower().strip().split()[0]
                if "analytic" in action:
                    action = "analytics_tool"
                elif "table" in action:
                    action = "table_search"
                elif "doc" in action:
                    action = "doc_search"
                elif "graph" in action:
                    action = "graph_search"

                print(f"\n Action: {action} | Input: {action_input[:80]}")

                if action in tools_dict:
                    observacion = tools_dict[action].invoke(action_input)
                    print(f"\n Observation: {str(observacion)[:300]}...")
                    contexto_acumulado += f"\n[{action} → '{action_input[:50]}']: {str(observacion)[:400]}\n"
                else:
                    # Fallback por palabras clave si la herramienta no se reconoce
                    consulta_lower = consulta_usuario.lower()
                    if any(p in consulta_lower for p in ["categoría", "categoria", "marca", "sucursal"]):
                        observacion = tools_dict["graph_search"].invoke(consulta_usuario)
                    elif any(p in consulta_lower for p in ["precio", "stock", "cuesta", "disponible"]):
                        observacion = tools_dict["table_search"].invoke(consulta_usuario)
                    elif any(p in consulta_lower for p in ["pago", "venta", "devolución", "estadística"]):
                        observacion = tools_dict["analytics_tool"].invoke(consulta_usuario)
                    else:
                        observacion = tools_dict["doc_search"].invoke(consulta_usuario)
                    contexto_acumulado += f"\n[fallback]: {str(observacion)[:400]}\n"

            except Exception as e:
                contexto_acumulado += f"\nError: {str(e)}\n"
        else:
            break

    # Si se agotaron las iteraciones, generar respuesta con contexto acumulado
    prompt_final = f"""Con esta información:
{contexto_acumulado}

Respondé brevemente en español la pregunta: {consulta_usuario}"""

    respuesta_final = generar(
        "Sos un asistente de ElectroHogar. Respondé en máximo 3 oraciones en español.",
        prompt_final,
        max_tokens=100
    )
    print(f"\n RESPUESTA FINAL: {respuesta_final}")
    historial_agente.append({
        "usuario": consulta_usuario,
        "respuesta": respuesta_final
    })
    return respuesta_final

print(" Agente ReAct listo")

 Agente ReAct listo


In [24]:
# CELDA 24: Pruebas Ejercicio 2

pruebas_ejercicio2 = [
    "¿Cómo uso mi licuadora para hacer smoothies?",
    "¿Cuáles son las licuadoras disponibles y cuánto cuestan?",
    "¿Qué opinan los usuarios de la cafetera?",
    "¿Qué productos hay en la categoría Cocina?",
    "¿Cuál es la distribución de métodos de pago?",
]

for i, prueba in enumerate(pruebas_ejercicio2, 1):
    historial_agente = []  # resetear historial entre pruebas
    print(f"\n{'#'*60}")
    print(f"PRUEBA {i}")
    agente_react(prueba)
    print()


############################################################
PRUEBA 1

CONSULTA: ¿Cómo uso mi licuadora para hacer smoothies?

 Iter 1:
Thought: Según el contexto proporcionado, parece que no hay ninguna consulta específica acerca del manejo o uso de una licuadora. Sin embargo, puedo intentarlo buscando reseñas generales de cómo usar las licuadoras.
Action: doc_search
Action Input: como usar licuadora para preparar smoothie

 Action: doc_search | Input: como usar licuadora para preparar smoothie

 Observation: [Score: 0.50] ### Base con Display LCD
**Código:** `BAS-P0004-004`

Base de abs de alto impacto. Temporizador programable, Control de temperatura, Interfaz intuitiva

### Sistema de Seguridad Integrado
**Código:** 
---
[Score: 0.38] ¿Qué voltaje requiere? El Licuadora funciona con 12V. El consumo es...

 Iter 2:
Thought: Según el contexto proporcionado hasta ahora, parece que hay una falta de detalles específicos acerca del manejo o las instrucciones detalladas para utilizar est

## Pruebas Manuales

In [25]:
# CELDA OPCIONAL — Prueba interactiva del chatbot
# Descomentar para probar manualmente

#resetear_historial()
#while True:
#     consulta = input("Usuario: ")
#     if consulta.lower() == "salir":
#         break
#     respuesta = chatbot(consulta)
#     print(f"Asistente: {respuesta}\n")

Historial reseteado
Usuario: ¿Qué lavarropas tienen disponibles?
   [Intención: tabular]
   [Consulta enriquecida: ¿Qué lavarropas tienen disponibles?]
   [Contexto:                            nombre  precio_usd  stock
Deluxe Lavarropas Automático 2024      191.93      2
            Lavarropas Automático      708.55    177
       Lavarropas Automático 3000     278...]
Asistente: Compacto Lavador Pónico Automático  
Lavadores Pónicos Automáticos están disponibles en dos precios y cantidades diferentes.

Usuario: SALIR


In [28]:
# CELDA OPCIONAL — Prueba interactiva del agente ReAct
# Descomentar para probar manualmente

#historial_agente = []
#consulta = input("Consulta para el agente: ")
#agente_react(consulta)

Consulta para el agente: ¿Qué productos hay en Climatización?

CONSULTA: ¿Qué productos hay en Climatización?

 Iter 1:
Thought: Según tu pregunta inicial, necesito saber qué tipos de climatizadores o dispositivos relacionados están disponibles. Usando el método adecuado sería útil primero ver las categorías.
Action: graph_search
Action Input: categoría climatización

 Action: graph_search | Input: categoría climatización

 Observation: Resultados del grafo:
{'resultado': 'Compacto Ventilador de Torre', 'relacion': 'Climatización', 'precio': 1217.47}
{'resultado': 'Deluxe Ionizador', 'relacion': 'Climatización', 'precio': 1390.51}
{'resultado': 'Max Ionizador', 'relacion': 'Climatización', 'precio': 1585.04}
{'resultado': 'Advanced...

 Iter 2:
Thought: Con base en el resultado obtenido anteriormente que muestra los diferentes tipos de ventiladores e ionizadores relacionados con Climatización, puedo inferir las principales opciones disponibles.
Final Answer: En categoría Climatización 

'En categoría Climatización se encuentran varios modelos como Compacto Ventilador de Torre, Deluxe Ionizador, Max Ionizador, Advanced Ionizador Plus y Ionizador X. Estos son algunos ejemplos representativos pero no necesariamente todos los productos existentes dentro esta categoría específica.'